Part 2: Board Game Recommendation

In [ ]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors
import requests
import xml.etree.ElementTree as ET
import time
import pandas as pd

# Replace 'your_file.csv' with the actual path or name of your CSV file
df = pd.read_csv('boardgames_ranks.csv')


def get_bgg_game_info_batch(game_ids):
    games_data = []
    # BGG API recommends batches of a few hundred to avoid timeouts
    batch_size = 300 
    
    for i in range(0, len(game_ids), batch_size):
        batch = game_ids[i:i + batch_size]
        ids_str = ",".join(map(str, batch))
        
        # BGG XML API2 endpoint
        url = f"https://boardgamegeek.com/xmlapi2/thing?id={ids_str}&stats=1"
        headers = {"User-Agent": "JupyterBGGDataFetcher/1.0"}
        
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            root = ET.fromstring(response.content)
            
            for item in root.findall('item'):
                def get_val(element):
                    return element.attrib.get('value') if element is not None else None
                
                ratings = item.find('statistics/ratings')
                
                games_data.append({
                    "id": int(item.attrib.get('id')),
                    "min_players": get_val(item.find('minplayers')),
                    "max_players": get_val(item.find('maxplayers')),
                    "playtime_mins": get_val(item.find('playingtime')),
                    "min_age": get_val(item.find('minage')),
                    "complexity_weight": get_val(ratings.find('averageweight') if ratings is not None else None),
                })
        else:
            print(f"Error fetching batch {i}-{i+batch_size}: HTTP {response.status_code}")
            
        # Politeness delay between batch requests
        time.sleep(2) 
        
    return pd.DataFrame(games_data)

# 1. Extract IDs from your existing baseGames DataFrame
ids_to_fetch = baseGames['id'].tolist()

# 2. Fetch the extra stats (Note: this might take a minute or two depending on dataset size)
print(f"Fetching BGG stats for {len(ids_to_fetch)} games in batches...")
bgg_stats_df = get_bgg_game_info_batch(ids_to_fetch)

# 3. Ensure both ID columns are integers for a clean merge
bgg_stats_df['id'] = bgg_stats_df['id'].astype(int)
baseGames['id'] = baseGames['id'].astype(int)

# 4. Merge the new stats into the baseGames DataFrame
extended_df = pd.merge(baseGames, bgg_stats_df, on='id', how='left')

# 5. Convert the new columns to numeric types (handling any missing values gracefully)
numeric_cols = ['min_players', 'max_players', 'playtime_mins', 'min_age', 'complexity_weight']
for col in numeric_cols:
    extended_df[col] = pd.to_numeric(extended_df[col], errors='coerce')

# Display the newly merged DataFrame
extended_df.head()


{'Game': 'Clank!: A Deck-Building Adventure', 'Estimated Rank': 121, 'Actual Rank': np.int64(105)}
